# Notebook 2 — Meal Completeness Score (§3.2.1)
**Demonstrates the meal blueprint scoring function across 8 real cart states, dynamic re-scoring on item removal, and how the score gates the MMoE expert routing.**

> CART-SYNCZ · Team KVK · Zomathon 2025


In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("=" * 60)
print("CART-SYNCZ  |  Notebook 2: Meal Completeness Score")
print("=" * 60)


## MEAL BLUEPRINTS  (§3.2.1)


In [ ]:
# ─────────────────────────────────────────────
MEAL_BLUEPRINTS = {
    "North Indian":  ["Main", "Bread", "Side", "Beverage", "Dessert"],
    "Mughlai":       ["Main", "Side",  "Bread", "Beverage", "Dessert"],
    "South Indian":  ["Main", "Rice",  "Side",  "Beverage", "Dessert"],
    "Chinese":       ["Main", "Rice/Noodles", "Side", "Beverage", "Dessert"],
    "Continental":   ["Starter", "Main", "Side", "Beverage", "Dessert"],
    "Street Food":   ["Main", "Side", "Beverage", "Extras"],
}

# Known items and their components
ITEM_META = {
    "Chicken Biryani":       ("Mughlai",      "Main"),
    "Mutton Biryani":        ("Mughlai",      "Main"),
    "Salan":                 ("Mughlai",      "Side"),
    "Raita":                 ("Mughlai",      "Side"),
    "Sheermal":              ("Mughlai",      "Bread"),
    "Pepsi":                 ("Mughlai",      "Beverage"),
    "Phirni":                ("Mughlai",      "Dessert"),
    "Butter Chicken":        ("North Indian", "Main"),
    "Paneer Butter Masala":  ("North Indian", "Main"),
    "Naan":                  ("North Indian", "Bread"),
    "Roti":                  ("North Indian", "Bread"),
    "Dal Makhani":           ("North Indian", "Side"),
    "Gulab Jamun":           ("North Indian", "Dessert"),
    "Lassi":                 ("North Indian", "Beverage"),
    "Masala Dosa":           ("South Indian", "Main"),
    "Idli":                  ("South Indian", "Main"),
    "Vada":                  ("South Indian", "Side"),
    "Sambar Rice":           ("South Indian", "Rice"),
    "Filter Coffee":         ("South Indian", "Beverage"),
    "Kesari Bath":           ("South Indian", "Dessert"),
}

def infer_dominant_cuisine(cart_items):
    """Find the most common cuisine in the cart."""
    cuisines = [ITEM_META[i][0] for i in cart_items if i in ITEM_META]
    if not cuisines:
        return "North Indian"
    return max(set(cuisines), key=cuisines.count)

def meal_completeness_score(cart_items):
    """
    Computes a score in [0, 1] representing how 'complete' the meal is.
    1.0 = all blueprint components present.
    """
    cuisine = infer_dominant_cuisine(cart_items)
    blueprint = MEAL_BLUEPRINTS.get(cuisine, MEAL_BLUEPRINTS["North Indian"])
    components_in_cart = set()
    for item in cart_items:
        if item in ITEM_META:
            components_in_cart.add(ITEM_META[item][1])
    present = len(components_in_cart.intersection(set(blueprint)))
    score = present / len(blueprint)
    return round(score, 2), cuisine, list(components_in_cart), blueprint

def missing_components(cart_items):
    score, cuisine, present_comps, blueprint = meal_completeness_score(cart_items)
    missing = [c for c in blueprint if c not in present_comps]
    return missing

def recommend_action(score):
    """Decide what the CSAO rail should do based on score."""
    if score < 0.4:
        return "🎯 Target meal-completion items  (Main course missing or very sparse)"
    elif score < 0.8:
        return "🥗 Fill missing components  (Side / Bread / Beverage)"
    else:
        return "💎 Pivot to value-add extras  (Dessert, premium add-ons, upsells)"


## TEST CASES


In [ ]:
# ─────────────────────────────────────────────
print("\n" + "─"*60)
print("  DEMO: Meal Completeness Scoring on Real Cart Examples")
print("─"*60)

TEST_CARTS = [
    {
        "label": "Cart A — Biryani only (single item, classic cold-start scenario)",
        "items": ["Chicken Biryani"],
    },
    {
        "label": "Cart B — Biryani + Salan (partial, 2-item)",
        "items": ["Chicken Biryani", "Salan"],
    },
    {
        "label": "Cart C — Biryani + Salan + Phirni (3-item, nearly complete)",
        "items": ["Chicken Biryani", "Salan", "Phirni"],
    },
    {
        "label": "Cart D — Biryani + Salan + Pepsi + Phirni (approaching complete)",
        "items": ["Chicken Biryani", "Salan", "Pepsi", "Phirni"],
    },
    {
        "label": "Cart E — Full Mughlai meal (all components)",
        "items": ["Chicken Biryani", "Salan", "Sheermal", "Pepsi", "Phirni"],
    },
    {
        "label": "Cart F — Butter Chicken only (North Indian)",
        "items": ["Butter Chicken"],
    },
    {
        "label": "Cart G — Butter Chicken + Naan + Dal Makhani (North Indian)",
        "items": ["Butter Chicken", "Naan", "Dal Makhani"],
    },
    {
        "label": "Cart H — Full North Indian meal",
        "items": ["Butter Chicken", "Naan", "Dal Makhani", "Lassi", "Gulab Jamun"],
    },
]

for cart in TEST_CARTS:
    score, cuisine, present_comps, blueprint = meal_completeness_score(cart["items"])
    missing = missing_components(cart["items"])
    action = recommend_action(score)

    bar_filled = int(score * 20)
    bar = "█" * bar_filled + "░" * (20 - bar_filled)

    print(f"\n  {cart['label']}")
    print(f"  Cart:     {cart['items']}")
    print(f"  Cuisine:  {cuisine}")
    print(f"  Score:    {score:.2f}  [{bar}]  {int(score*100)}%")
    print(f"  Present:  {present_comps}")
    print(f"  Missing:  {missing}")
    print(f"  CSAO:     {action}")


## DYNAMIC UPDATE: item removed from cart


In [ ]:
# ─────────────────────────────────────────────
print("\n" + "─"*60)
print("  DEMO: Cart item removed → score re-computed in <15ms")
print("─"*60)

cart_states = [
    ["Chicken Biryani"],
    ["Chicken Biryani", "Salan"],
    ["Chicken Biryani", "Salan", "Pepsi"],
    ["Chicken Biryani", "Salan", "Pepsi", "Phirni"],
    # User removes Salan
    ["Chicken Biryani", "Pepsi", "Phirni"],
]
labels = ["Add Biryani", "Add Salan", "Add Pepsi", "Add Phirni", "Remove Salan ←"]

print(f"\n  {'Step':<18} {'Cart Size':<12} {'Score':<8} {'Action'}")
print("  " + "-"*70)
for label, cart in zip(labels, cart_states):
    score, *_ = meal_completeness_score(cart)
    action = recommend_action(score)
    print(f"  {label:<18} {len(cart):<12} {score:<8.2f} {action}")


## SCORE → GATING NETWORK DEMO


In [ ]:
# ─────────────────────────────────────────────
print("\n" + "─"*60)
print("  DEMO: How score feeds into MMoE gating network")
print("─"*60)
print("""
  The meal_completeness_score is a direct input feature to the 
  MMoE gating network (§4.3.3):

  Low completeness (< 0.8):
    → Gating weight shifts toward P(Accept) expert
    → Model prioritises recommending missing meal components
    → e.g., Biryani-only cart → high weight on Side/Bread recs

  High completeness (≥ 0.8):
    → Gating weight shifts toward E(AOV Lift) expert  
    → Model pivots to premium upsells and value-add extras
    → e.g., complete Mughlai meal → premium Lassi, Kulfi, etc.

  Score range  |  P(Accept) gate weight  |  AOV Lift gate weight
  ─────────────────────────────────────────────────────────────""")

for score_val in [0.2, 0.4, 0.6, 0.8, 1.0]:
    # Simulated gating logic: p_weight decreases as completeness rises
    p_weight = max(0.3, 0.9 - score_val * 0.7)
    aov_weight = 1 - p_weight
    bar_p   = "█" * int(p_weight * 20)
    bar_aov = "█" * int(aov_weight * 20)
    print(f"  {score_val:.1f}         |  {p_weight:.2f}  {bar_p:<20}  |  {aov_weight:.2f}  {bar_aov}")

print("\n" + "=" * 60)
print("✓  Meal completeness score demo complete.")
print("   Cart scoring ✓  |  Dynamic re-scoring ✓  |  MMoE gating ✓")
print("=" * 60)
